# 🌳 Achando a Maior Árvore do Brasil (com LiDAR!)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ia-para-jovens-curiosos/achando_a_maior_arvore_do_brasil_colab/blob/main/arvrona.ipynb)

Há alguns anos, foram feitos ~1000 vôos sobre a Amazônia carregando uma máquina chamada **sensor LiDAR**. Esse sensor disparou 100.000 pulsos laser por segundo em direção ao chão e mediu o tempo que cada um levou para voltar, criando uma **nuvem de pontos**: milhões de coordenadas X, Y, Z (posição e altura) espalhadas pela floresta.

![Avião usado nos voos de LiDAR](https://raw.githubusercontent.com/ia-para-jovens-curiosos/achando_a_maior_arvore_do_brasil_colab/main/images/aeronave.png)

<small>*Fig. 1 - O avião usado nos voos de LiDAR.*</small>

![Equipamento de LiDAR instalado dentro do avião](https://raw.githubusercontent.com/ia-para-jovens-curiosos/achando_a_maior_arvore_do_brasil_colab/main/images/maquina_lidar.png)

<small>*Fig. 2 - O equipamento de LiDAR instalado dentro do avião.*</small>

![Mapa com os ~1000 voos de LiDAR sobre a Amazônia](https://raw.githubusercontent.com/ia-para-jovens-curiosos/achando_a_maior_arvore_do_brasil_colab/main/images/mapa_voos.png)

<small>*Fig. 3 - Mapa com os ~1000 voos de LiDAR sobre a Amazônia.*</small>

Numa dessas nuvens de pontos está escondida a **maior árvore do Brasil**. Neste notebook, você vai construir um algoritmo que procura essa árvore sozinho, **passo a passo**, e vai poder ver a busca acontecendo em gráficos.

O passo a passo é sempre o mesmo:

1. **Carregar a nuvem** de pontos (`carregar_nuvem`)
2. **Buscar a árvore campeã**, vendo cada passo da busca (`buscar_maior_arvore`)
3. **Mostrar o resultado** final (`mostrar_resultado`)

Execute as células de cima para baixo, uma de cada vez (clique na célula e pressione `Shift + Enter`, ou clique no botão ▶️ ao lado da célula).

## Passo 0 — Preparar o ambiente 🧰

Se você estiver no **Google Colab**, a célula abaixo baixa o projeto do GitHub (o código, os dados e as imagens) e instala o leitor de arquivos LiDAR. Isso leva alguns segundos e só precisa ser feito uma vez.

Se você estiver rodando no seu computador, ela não faz nada — o projeto já está aí.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    PASTA_DO_PROJETO = "/content/achando_a_maior_arvore_do_brasil_colab"
    if not os.path.exists(PASTA_DO_PROJETO):
        subprocess.run(["git", "clone", "-q", "https://github.com/ia-para-jovens-curiosos/achando_a_maior_arvore_do_brasil_colab.git", PASTA_DO_PROJETO], check=True)
    os.chdir(PASTA_DO_PROJETO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laspy", "lazrs"], check=True)

print("Ambiente preparado em", os.getcwd())

In [ ]:
%matplotlib inline
from achando_a_maior_arvore import carregar_nuvem, buscar_maior_arvore, mostrar_resultado

print("Tudo pronto! Vamos começar 🚀")

## Passo 1 — Carregar a nuvem de pontos 📥

Vamos ler o arquivo `data/nuvem_de_pontos_aula.laz`, um pedaço (uma "faixa") da nuvem de pontos original, recortado especialmente para esta aula. Mesmo assim, ele tem mais de seis milhões de pontos!

In [ ]:
nuvem = carregar_nuvem()

## Passo 2 — Buscar a árvore campeã 🔍

Como o algoritmo decide, sozinho, qual ponto é realmente o topo de uma árvore?

1. Ele olha para o **ponto mais alto** de toda a nuvem.
2. Ao redor desse ponto, ele imagina um **hemisfério** (uma "meia bola") de **50 metros de diâmetro** (25 m de raio), virado para baixo, com a borda bem no centro do ponto escolhido.
3. Ele **conta quantos pontos** existem dentro desse hemisfério.
4. Uma árvore de verdade tem uma copa cheia de galhos e folhas por baixo do topo — ou seja, deve haver **pelo menos 300 pontos** ali dentro.
   - Se houver 300 pontos ou mais: ✅ esse ponto é aceito como a árvore!
   - Se houver menos: ❌ o ponto é descartado (pode ser um passarinho, uma nuvem baixa ou um erro do laser), e o algoritmo tenta o **próximo ponto mais alto**.
5. Ele repete esse processo até encontrar um ponto que passe no teste.

A célula abaixo mostra **cada passo** dessa busca: um gráfico de perfil lateral da floresta e uma vista de cima, com o ponto testado e o círculo do hemisfério desenhados em vermelho (rejeitado) ou verde (aceito).

In [ ]:
campea = buscar_maior_arvore(nuvem)

## Passo 3 — Ver o resultado 🏆

Agora vamos ver a árvore campeã destacada dentro da nuvem de pontos inteira.

In [ ]:
mostrar_resultado(nuvem, campea)

## Para pensar 🤔

- Por que testar só a **metade de baixo** do hemisfério, e não a bola inteira?
- O que aconteceria se o número mínimo de pontos fosse bem menor, tipo 10? E se fosse bem maior, tipo 10.000?
- O que aconteceria se o hemisfério fosse bem menor (5 m de raio) ou bem maior (100 m de raio)?

Tente mudar os valores de `RAIO_DO_HEMISFERIO_METROS` e `MINIMO_DE_PONTOS_NO_HEMISFERIO` na célula abaixo e rode-a de novo para descobrir! (Para a busca ficar rápida, ela não desenha os gráficos de cada passo.)

In [ ]:
import achando_a_maior_arvore as arvore

arvore.RAIO_DO_HEMISFERIO_METROS = 25.0
arvore.MINIMO_DE_PONTOS_NO_HEMISFERIO = 300

campea = buscar_maior_arvore(nuvem, mostrar_passos=False)
mostrar_resultado(nuvem, campea)

### Veja as imagens da nuvem de pontos com a árvore de perfil e vista de cima

In [ ]:
from glob import glob
from IPython.display import Image, display

for caminho in sorted(glob("images/arvrona*.png")):
    display(Image(filename=caminho))